In [1]:
#Cell 1--Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss,
    log_loss
)

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
#Cell 2--Load Data
train_path = '/Users/codylewis/Desktop/AIHC 5615/Week 8/heart-disease-complete-train.csv'
test_path  = '/Users/codylewis/Desktop/AIHC 5615/Week 8/heart-disease-complete-test.csv'

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (576, 15)
Test shape: (144, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,clinic,hd
0,49.0,0.0,3.0,160.0,180.0,0.0,0.0,156.0,0.0,1.0,2.0,NaN,NaN,hung,1
1,62.0,0.0,4.0,124.0,209.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,clev,0
2,60.0,1.0,3.0,115.0,0.0,NaN,0.0,143.0,0.0,2.4,1.0,NaN,NaN,swit,1
3,65.0,1.0,4.0,160.0,0.0,1.0,1.0,122.0,0.0,NaN,NaN,NaN,7.0,swit,1
4,34.0,1.0,1.0,118.0,182.0,0.0,2.0,174.0,0.0,0.0,1.0,0.0,3.0,clev,0


In [3]:
#Cell 3--Identify Variable Types
target = "hd"

# Explicit numeric columns based on dataset description
num_cols = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

# Explicit categorical columns
cat_cols = [
    "sex", "cp", "fbs", "restecg", "exang",
    "slope", "thal", "clinic", "ca"
]

print("Numeric predictors:", num_cols)
print("Categorical predictors:", cat_cols)



Numeric predictors: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical predictors: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'clinic', 'ca']


In [4]:
#Cell 4--Handle Missing Values
train_df_clean = train_df.copy()
test_df_clean  = test_df.copy()

# Numeric: median
for c in num_cols:
    median_val = train_df_clean[c].median()
    train_df_clean[c] = train_df_clean[c].fillna(median_val)
    test_df_clean[c]  = test_df_clean[c].fillna(median_val)

# Categorical: mode
for c in cat_cols:
    mode_val = train_df_clean[c].mode()[0]
    train_df_clean[c] = train_df_clean[c].fillna(mode_val)
    test_df_clean[c]  = test_df_clean[c].fillna(mode_val)

train_df_clean.isna().sum()


age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
clinic      0
hd          0
dtype: int64

In [5]:
# Show full cleaned training data with no truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

train_df_clean


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,clinic,hd
0,49.0,0.0,3.0,160.0,180.0,0.0,0.0,156.0,0.0,1.0,2.0,0.0,3.0,hung,1
1,62.0,0.0,4.0,124.0,209.0,0.0,0.0,163.0,0.0,0.0,1.0,0.0,3.0,clev,0
2,60.0,1.0,3.0,115.0,0.0,0.0,0.0,143.0,0.0,2.4,1.0,0.0,3.0,swit,1
3,65.0,1.0,4.0,160.0,0.0,1.0,1.0,122.0,0.0,0.2,2.0,0.0,7.0,swit,1
4,34.0,1.0,1.0,118.0,182.0,0.0,2.0,174.0,0.0,0.0,1.0,0.0,3.0,clev,0
5,55.0,1.0,2.0,130.0,262.0,0.0,0.0,155.0,0.0,0.0,1.0,0.0,3.0,clev,0
6,57.0,1.0,4.0,152.0,274.0,0.0,0.0,88.0,1.0,1.2,2.0,1.0,7.0,clev,1
7,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,clev,1
8,35.0,1.0,4.0,126.0,282.0,0.0,2.0,156.0,1.0,0.0,1.0,0.0,7.0,clev,1
9,51.0,0.0,3.0,130.0,256.0,0.0,2.0,149.0,0.0,0.5,1.0,0.0,3.0,clev,0


In [ ]:
#Cell 5--Outlier Handling
train_clip = train_df_clean.copy()
test_clip  = test_df_clean.copy()

low = train_clip[num_cols].quantile(0.01)
high = train_clip[num_cols].quantile(0.99)

for c in num_cols:
    train_clip[c] = train_clip[c].clip(lower=low[c], upper=high[c])
    test_clip[c]  = test_clip[c].clip(lower=low[c], upper=high[c])

train_clip.describe()


In [ ]:
#Cell 6--Define X and Y for modeling
X_train = train_clip.drop(columns=[target])
y_train = train_clip[target]

X_test  = test_clip.copy()


In [ ]:
#Cell 7-- Preprocessing pipeline for modeling

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify numeric & categorical columns
numeric_cols = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

categorical_cols = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "thal", "clinic", "ca"
]

# ColumnTransformer for scaling numeric and one-hot encoding categorical
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


In [ ]:
#Cell 8-- Fit Candidate models
full_model = Pipeline([
    ("prep", preprocess),
    ("logreg", LogisticRegression(max_iter=500, penalty=None))
])

full_model.fit(X_train, y_train)

full_pred = full_model.predict_proba(X_train)[:, 1]



In [ ]:
# Clean data for Statsmodels forward selection

# 1. Make a copy
train_fs = train_clip.copy()

# 2. Convert numeric-looking strings to actual numbers
for col in num_cols:
    train_fs[col] = pd.to_numeric(train_fs[col], errors="coerce")

# 3. Convert categorical columns to dtype category
for col in cat_cols:
    train_fs[col] = train_fs[col].astype("category")

# 4. Re-align y
y_fs = train_fs["hd"]


In [ ]:
#Cell 9--Forward Selection to Find 6 Best Predictors
import statsmodels.api as sm

candidate_predictors = num_cols + cat_cols
selected = []
remaining = candidate_predictors.copy()

best_aic = np.inf

for _ in range(6):

    scores = []
    for var in remaining:
        trial_vars = selected + [var]
        
        # *** USE CLEANED DATASET ***
        X_trial = pd.get_dummies(train_fs[trial_vars], drop_first=True)
        
        # Convert to numeric array to avoid dtype object
        X_trial = X_trial.apply(pd.to_numeric, errors="coerce").astype(float)
        
        # Add intercept
        X_trial = sm.add_constant(X_trial)

        # *** USE CLEANED TARGET ***
        model = sm.GLM(y_fs.astype(float), X_trial, family=sm.families.Binomial())
        result = model.fit()

        scores.append((result.aic, var))
    
    scores.sort()
    best_aic, best_var = scores[0]
    selected.append(best_var)
    remaining.remove(best_var)

print("Selected predictors:", selected)



In [ ]:
#Cell 10--Six-Predictor Logistic Regression
six_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), [c for c in selected if c in num_cols]),
        ("cat", OneHotEncoder(drop="first"), [c for c in selected if c in cat_cols])
    ]
)

six_model = Pipeline([
    ("prep", six_preprocess),
    ("logreg", LogisticRegression(max_iter=500))
])

six_model.fit(X_train[selected], y_train)

six_pred = six_model.predict_proba(X_train[selected])[:, 1]

metrics_six = {
    "Accuracy": accuracy_score(y_train, six_model.predict(X_train[selected])),
    "AUC": roc_auc_score(y_train, six_pred),
    "Brier": brier_score_loss(y_train, six_pred),
    "LogLoss": log_loss(y_train, six_pred)
}

metrics_six


In [ ]:
#Cell 11--Elastic Net Logistic Regression (FINAL MODEL)
final_model = Pipeline([
    ("prep", preprocess),
    ("logreg", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        max_iter=2000
    ))
])

final_model.fit(X_train, y_train)

final_pred_train = final_model.predict_proba(X_train)[:, 1]

metrics_final = {
    "Accuracy": accuracy_score(y_train, final_model.predict(X_train)),
    "AUC": roc_auc_score(y_train, final_pred_train),
    "Brier": brier_score_loss(y_train, final_pred_train),
    "LogLoss": log_loss(y_train, final_pred_train)
}

metrics_final



In [ ]:
#Cell 12-- Cross-validation
lambdas = np.logspace(-4, 2, 20)
cv_scores = []

for C in 1 / lambdas:
    model = Pipeline([
        ("prep", preprocess),
        ("logreg", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=C,
            max_iter=2000
        ))
    ])
    
    score = -np.mean(
        cross_val_score(model, X_train, y_train, cv=5, scoring="neg_log_loss")
    )
    cv_scores.append(score)

plt.figure(figsize=(8,5))
plt.semilogx(lambdas, cv_scores, marker="o")
plt.xlabel("λ (regularization strength)")
plt.ylabel("Cross-validated Log-Loss")
plt.title("Elastic Net Logistic Regression — CV Performance Curve")
plt.grid(True)
plt.show()


In [ ]:
#Cell 13--Test Predictions

test_pred_prob = final_model.predict_proba(X_test)[:, 1]
test_pred_binary = (test_pred_prob >= 0.5).astype(int)

pred_df = pd.DataFrame({
    "probability": test_pred_prob,
    "prediction": test_pred_binary
})

pred_df.head()



In [ ]:
#Cell 14--Save Prediction CSV
pred_df.to_csv("cody_predictions.csv", index=False)
print("Saved cody_predictions.csv!")


In [ ]:
#Cell 15- Print Final
print(final_model)
